In [16]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.6 MB/s eta 0:00:00


In [22]:
from datasets import Dataset
import evaluate
import os
from transformers import TrainingArguments, Trainer, AutoModelForSequenceClassification, AutoTokenizer
import pandas as pd
import torch


In [ ]:
columns = ["question_translated", "context", "answerable", "lang"]
train_df = pd.read_csv("train_translated.csv")
val_df = pd.read_csv("validation_translated.csv")
train_df = train_df[columns]
val_df   = val_df[columns]


train_dataset = Dataset.from_pandas(train_df)
val_dataset   = Dataset.from_pandas(val_df)


In [ ]:

model_name = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)


def preprocess(examples):
    texts = [q + " [SEP] " + c for q, c in zip(examples["question_translated"], examples["context"])]
    tokenized = tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=512
    )
    tokenized["labels"] = [int(a) for a in examples["answerable"]]

    tokenized["lang"] = examples["lang"]
    return tokenized


train_dataset = train_dataset.map(preprocess, batched=True)
val_dataset   = val_dataset.map(preprocess, batched=True)


train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels", "lang"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels", "lang"])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
 

Map:   0%|          | 0/6335 [00:00<?, ? examples/s]

Map:   0%|          | 0/1155 [00:00<?, ? examples/s]

In [ ]:
languages = ["ar", "ko", "te"]

train_by_lang = {lang: train_dataset.filter(lambda x: x["lang"].lower() == lang) for lang in languages}
val_by_lang   = {lang: val_dataset.filter(lambda x: x["lang"].lower() == lang) for lang in languages}


Filter:   0%|          | 0/6335 [00:00<?, ? examples/s]

Filter:   0%|          | 0/6335 [00:00<?, ? examples/s]

Filter:   0%|          | 0/6335 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1155 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1155 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1155 [00:00<?, ? examples/s]

In [ ]:

columns_to_keep = ["input_ids", "attention_mask", "labels"]

for lang in languages:
    train_by_lang[lang] = train_by_lang[lang].remove_columns(
        [c for c in train_by_lang[lang].column_names if c not in columns_to_keep]
    )
    val_by_lang[lang] = val_by_lang[lang].remove_columns(
        [c for c in val_by_lang[lang].column_names if c not in columns_to_keep]
    )

    train_by_lang[lang].set_format(type="torch", columns=columns_to_keep)
    val_by_lang[lang].set_format(type="torch", columns=columns_to_keep)


In [ ]:


models_by_lang = {
    lang: AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2
    )
    for lang in languages
}

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to

In [10]:


for lang in languages:
    train_dataset = train_by_lang[lang]
    print(f"Training for lang: {lang}")
    training_args = TrainingArguments(
        output_dir=f'./results_{lang}',
        per_device_train_batch_size=8,
        num_train_epochs=3,
        learning_rate=2e-5,
        save_total_limit=1,
        logging_steps=50,
        report_to=[]
    )

    trainer = Trainer(
        model=models_by_lang[lang],
        args=training_args,
        train_dataset=train_dataset,
        tokenizer=tokenizer
    )

    trainer.train()


Training for lang: ar


/tmp/ipython-input-3849044985.py:14: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Step,Training Loss
50,0.394600
100,0.345100
150,0.269500
200,0.284100
250,0.115600
300,0.138600
350,0.079700
400,0.139800
450,0.092100
500,0.063500


Training for lang: ko


/tmp/ipython-input-3849044985.py:14: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Step,Training Loss
50,0.287900
100,0.137800
150,0.103500
200,0.096000
250,0.073000
300,0.054400
350,0.074900
400,0.034600
450,0.080900
500,0.064300


Training for lang: te


/tmp/ipython-input-3849044985.py:14: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Step,Training Loss
50,0.301900
100,0.179300


Step,Training Loss
50,0.301900
100,0.179300
150,0.107100
200,0.136000
250,0.105000
300,0.148600
350,0.034200
400,0.062200
450,0.066700
500,0.074300


In [ ]:
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = torch.argmax(torch.tensor(logits), dim=-1)
    return metric.compute(predictions=predictions, references=labels)

for lang in languages:
    print(f"Evaluating model for language: {lang}")

    model = models_by_lang[lang]  
    val_dataset_lang = val_by_lang[lang]

    val_dataset_lang.set_format(
        type="torch",
        columns=["input_ids", "attention_mask", "labels"]
    )

    eval_args = TrainingArguments(
        output_dir=f"./eval_{lang}",
        per_device_eval_batch_size=8,
        do_train=False,
        do_eval=True,
        logging_dir=f"./logs_{lang}",
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=eval_args,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics
    )

    eval_result = trainer.evaluate(eval_dataset=val_dataset_lang)
    print(f"Validation results for {lang}: {eval_result}")


Evaluating model for language: ar


/tmp/ipython-input-3718049710.py:32: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Validation results for ar: {'eval_loss': 0.1533946543931961, 'eval_model_preparation_time': 0.0025, 'eval_accuracy': 0.9710843373493976, 'eval_runtime': 19.1703, 'eval_samples_per_second': 21.648, 'eval_steps_per_second': 2.713}
Evaluating model for language: ko


/tmp/ipython-input-3718049710.py:32: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Validation results for ko: {'eval_loss': 0.11466605216264725, 'eval_model_preparation_time': 0.0028, 'eval_accuracy': 0.9719101123595506, 'eval_runtime': 16.8231, 'eval_samples_per_second': 21.161, 'eval_steps_per_second': 2.675}
Evaluating model for language: te


/tmp/ipython-input-3718049710.py:32: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Validation results for te: {'eval_loss': 0.6990228295326233, 'eval_model_preparation_time': 0.0047, 'eval_accuracy': 0.8723958333333334, 'eval_runtime': 18.1808, 'eval_samples_per_second': 21.121, 'eval_steps_per_second': 2.64}


In [28]:
print(train_by_lang)
print(val_by_lang)

{'ar': Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 2558
}), 'ko': Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 2422
}), 'te': Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1355
})}
{'ar': Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 415
}), 'ko': Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 356
}), 'te': Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 384
})}
